# FedAsync with staleness scaling (notebook 03, fixed)

Like plain async FL, but **old updates count less**. Based on Xie et al., *Asynchronous Federated Optimization* (2019).

**How it works**

1. Same as plain async: every update is applied when it arrives.
2. Step size: `rate = server_rate / (1 + staleness) ^ power`.
3. An update made on a model 4 versions old moves the global model 5× less.

> **Correction:** the original notebook 03 put the staleness factor into a *weight* that `apply_delta` then normalised away (w / w = 1), so it behaved **exactly like notebook 02**. Here the factor scales the **rate**, so it really takes effect.

> **Note:** `power` controls how harsh the penalty is: 0 means no penalty, larger values punish stale updates more.

## 1. Load the simulator

Gives this notebook everything from `simulator.ipynb`: the settings, `Approach`, `fedavg`, `apply_delta` and `run`.

In [1]:
%run ../simulator.ipynb

## 2. The approach

Parameters are the class attributes at the top. Change them here, or pass them when running (see the tip below).

In [2]:
class FedAsyncStaleness(Approach):
    name = "fedasync_staleness"
    label = "Async staleness weighted"          # shown in plot legends
    server_rate = 0.5
    power = 1.0                   # staleness penalty strength

    def on_update(self, server, update):
        rate = self.server_rate / (1 + update.staleness) ** self.power
        new_model = apply_delta(server.model, [update], [1.0], rate)
        server.set_model(new_model, [update])

## 3. Run it

Runs every seed listed in `settings.yaml` and saves the results to `results/`. Then open **`compare.ipynb`** to see it next to the other approaches.

> **Tip:** to try a variant without editing the class, run e.g. `run(FedAsyncStaleness(power=0.5))`. It is saved as a separate run.

In [3]:
results = run(FedAsyncStaleness())

fedasync_staleness           seed=42   val= 92.40%  acc= 93.35%  time=   560.2s  updates=  100  worst10%= 92.21%  wall=6.3s


## 4. Quick look

The final evaluation for each seed:

In [4]:
results.groupby("seed").tail(1)[["seed", "time", "updates", "accuracy", "loss", "worst10_client_acc", "mean_staleness"]].round(4)

,seed,time,updates,accuracy,loss,worst10_client_acc,mean_staleness
47,42,560.1644,100,0.9335,0.2091,0.9221,8.0
